In [ ]:
%load_ext autoreload
%autoreload 2
from lab import *
import numpy as np

from collections.abc import Callable

def format(float):
    return f"{float:.3g}"

np.set_printoptions(formatter={"float": format})

In [ ]:
class Vector:
    def __init__(self, items: float | Scalar | list[float] | list[Scalar], name: str | None = None):
        _items = items if isinstance(items, list) else [items]
        self.length: int = len(_items)
        self.items: list[Scalar] = [
            (
                item
                if isinstance(item, Scalar)
                else Scalar(item, f"{name}_{i}" if isinstance(name, str) else str(item))
            )
            for (i, item) in enumerate(_items)
        ]
        self.name: str = str(_items[0]) if self.length == 1 else "vec"
        self.name = name if isinstance(name, str) else self.name

    def setError(self, error: float | list[float]):
        _error = error if isinstance(error, list) else [error]
        if len(_error) == self.length:
            for i, e in zip(self.items, _error):
                i.setError(e)
        elif len(_error) == 1:
            for i in self.items:
                i.setError(_error[0])
        else:
            raise ValueError(
                f"Error must be of length 1 or same as vector. Expected 1 or {self.length}, got {len(_error)}."
            )
        return self

    
    def broadcast(self, length: int):
        if self.length == length:
            return self
        elif self.length == 1:
            return Vector([self.items[0]] * length, self.name)
        else:
            raise ValueError(
                f'Self of length "{self.length}" not broadcastable to length: "{length}". Own length must be "1" or "{length}".'
            )

    def _value_repr(self):
        return f"[{" ".join([i._value_repr() for i in self.items])}]"

    def _error_repr(self):
        return f"[{" ".join([i._error_repr() for i in self.items])}]"

    def _name_repr(self):
        return [i._name_repr() for i in self.items]

    def __repr__(self):
        return f"Vector({self.name}: {self._value_repr()} ± {self._error_repr()})"

    def _rename_items(self, rename: Callable[[int, Scalar], str]):
        for i, item in enumerate(self.items):
            item.name = rename(i, item)
        return self

    def __add__(self, other):
        if not isinstance(other, Vector):
            other = Vector(other, str(other))

        if self.length >= other.length:
            out_value = [
                (s + o)() for (s, o) in zip(self.items, other.broadcast(self.length).items)
            ]
        else:
            out_value = [
                (s + o)() for (s, o) in zip(self.broadcast(other.length).items, other.items)
            ]

        out_name = f"{self.name} + {other.name}"
        out = Vector(out_value, out_name)
        return out

    def __radd__(self, other):
        if not isinstance(other, Vector):
            other = Scalar(other, name=str(other))

        out = self + other
        out.name = f"{other.name} + {self.name}"
        return out

    def __mul__(self, other):
        if not isinstance(other, Vector):
            other = Vector(other, str(other))

        if self.length >= other.length:
            out_value = [
                (s * o)() for (s, o) in zip(self.items, other.broadcast(self.length).items)
            ]
        else:
            out_value = [
                (s * o)() for (s, o) in zip(self.broadcast(other.length).items, other.items)
            ]

        out_name = f"{self.name} * {other.name}"
        out = Vector(out_value, out_name)
        return out

    def __rmul__(self, other):
        if not isinstance(other, Vector):
            other = Vector(other, name=str(other))

        out = self * other
        out.name = f"{other.name} * {self.name}"
        return out

    def __neg__(self):
        out = self * (-1)
        out.name = f"- {self.name}"
        return out

    def __sub__(self, other):
        if not isinstance(other, Vector):
            other = Vector(other, name=str(other))

        out = self + (-other)
        out.name = f"{self.name} - {other.name}"
        return out

    def __rsub__(self, other):
        if not isinstance(other, Vector):
            other = Vector(other, name=str(other))

        out = other + (-self)
        out.name = f"{other.name} - {self.name}"
        return out

    def __truediv__(self, other):
        if not isinstance(other, Vector):
            other = Vector(other, name=str(other))

        if self.length >= other.length:
            out_value = [
                (s / o)() for (s, o) in zip(self.items, other.broadcast(self.length).items)
            ]
        else:
            out_value = [
                (s / o)() for (s, o) in zip(self.broadcast(other.length).items, other.items)
            ]

        out_name = f"{self.name} / {other.name}"
        out = Vector(out_value, out_name)
        return out

    def __rtruediv__(self, other):
        if not isinstance(other, Vector):
            other = Vector(other, name=str(other))

        out = other / self
        out.name = f"{other.name} / {self.name}"
        return out

In [ ]:
x = Vector([1, 2], "x").setError(0.1)
x * x